# Setup and main-figure input requirements

This notebook checks the environment and shows the input data required by Figs. 1-5. The raw inventory comes from `config/raw_data_require.json`, the same configuration used by the raster loader. Obtain and prepare TIFFs using the manuscript's data-source information and `data/raw/README.md`; this notebook does not download data.

The bundled NPZ route can run without raw TIFFs. To start from TIFFs, supply the complete main-figure raster set and run the optional header checks below before running a figure notebook.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/analysis.json").exists())
sys.path.insert(0, str(ROOT / "src"))
from elephant_analysis.runtime import configure_runtime
configure_runtime()
DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"
FIGURES_ROOT = ROOT / "figures"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
from elephant_analysis.runtime import dependency_report
dependency_report()

In [ ]:
from elephant_analysis.data_inputs import describe_inputs, describe_raw_inputs
from elephant_analysis.artifacts import load_json
import pandas as pd

print("Bundled inputs for the default workflow:")
display(pd.DataFrame(describe_inputs(DATA_ROOT)))
requirements = load_json(ROOT / "config/raw_data_require.json")
raw_status = pd.DataFrame(describe_raw_inputs(DATA_ROOT, requirements))
print(f"Main-figure raw inputs: {len(raw_status)} required files")
display(raw_status[["variable", "file", "main_figures", "purpose", "input_scale", "available"]])
print(f"Present: {int(raw_status['available'].sum())}/{len(raw_status)}")
print("No TIFF download or pixel-array loading is performed by this inventory cell.")


## Optional raw-grid check

After placing all required TIFFs in `data/raw/`, set `CHECK_RAW_GRID=True` and run the next cell. It reads file headers only, checking dimensions, origin, resolution, bands and matching geographic coordinate reference systems. A failed or missing input is listed explicitly. It does not verify data units, time periods, scaling or how the study products were prepared.


In [ ]:
CHECK_RAW_GRID = False  # Enable after all 19 prepared TIFFs have been supplied.
if CHECK_RAW_GRID:
    raw_checks = pd.DataFrame(describe_raw_inputs(DATA_ROOT, requirements, check_grid=True))
    display(raw_checks[["file", "available", "grid_check"]])
    failed = raw_checks.loc[raw_checks["grid_check"] != "ok", "file"].tolist()
    if failed:
        raise ValueError("Raw inputs are missing or have incompatible grids: " + ", ".join(failed))
    print("All required raster headers match the configured grid. Verify source products and numerical scales before analysis.")
else:
    print("Raw-grid check disabled. The default bundled-input workflow remains available.")


## Run a figure notebook

**From raw rasters:** open notebook 01, 03, 04 or 05 and set the following in that notebook's first code cell:

```python
INPUT_MODE = "raw"
RUN_ANALYSIS = True
```

Keep `DATA_ROOT = ROOT / "data"` (or its existing `REPO_ROOT` equivalent), and select **Restart Kernel and Run All**. Each notebook is independent: notebook 00 does not change their settings, and Fig. 1 does not have to run first. The shared paired calculation currently loads all 19 main-figure rasters.

- Notebook 01 computes Fig. 1a,b from TIFFs; Fig. 1c,d still uses the supplied covariate/model-prediction bundle.
- Notebook 02 uses the supplied non-raster density observations; keep its reference mode.
- Notebooks 03 and 04 compute paired temperature and resistance analyses from TIFFs.
- For a new SEM fit in notebook 05, also set `COMPUTE_SEM=True`, install R packages `lavaan` and `jsonlite`, and configure `RSCRIPT` if needed. With `COMPUTE_SEM=False`, saved coefficients are reused only if the newly prepared SEM matrix exactly matches the reference.

Outputs are saved to `results/` and `figures/`. After a successful run, `RUN_ANALYSIS=False` can reuse that figure's saved results for plotting. To use the bundled prepared paired inputs instead, leave `INPUT_MODE="reference"` and `RUN_ANALYSIS=True`.
